In [1]:
!pip install ddgs trafilatura
!pip install openai-agents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.0/827.0 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 15.0 MB/s eta 0:00:00


In [2]:
import os
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura

from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from agents import Agent, Runner, function_tool

MODEL = "gpt-4.1-mini"

Step 1: Define the tools

In [3]:
@function_tool
def search_web(query:str):
  """Search the web using DuckDuckGo browser. Returns 3 results."""
  ddgs = DDGS()
  results = ddgs.text(query, max_results=3)
  print(f"  \u2705 search_web: Got results for {query}")
  return json.dumps(results, indent=2)

In [4]:
@function_tool
def fetch_url(url:str):
  """Fetch the content of a URL using trafilatura."""
  downloaded = trafilatura.fetch_url(url)
  if downloaded:
    text = trafilatura.extract(downloaded)
    if text:
      print(f"  \u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
      return text
  print(f" \u274C fetch_url:Failed to get content from {url[:60]}")
  return f"Could not extract text from {url}. Try a different source."

Step 2: The agents

In [5]:
RESEARCH_AGENT_PROMPT = """
You are a research specialist. Your job is to research a given topic and produce
a comprehensive research brief.


You have access to two tools:
- search_web: Search the web for information
- fetch_url: fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results - which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize
into a research brief

You MUST gather information from at least 3 distinct sources before delivering
your brief.
If you have fewer than 3 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working - search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call - sometimes just thinking through what you
have is the right move.
"""
research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools=[search_web,fetch_url]
)

In [6]:
WRITER_AGENT_PROMPT = """
You are a professional article writer.
You will receive a conversation history that includes two research briefs.
The orchestrator has already selected the best one — use ONLY the selected brief.
Ignore the rejected brief entirely.


Your job:
- Write a well-structured, engaging article based ONLY on the selected research brief
- Use a clear, conversational tone — write like a real blogger, not an academic
- Include relevant statistics and data points from the research
- Cite sources where appropriate using inline links
- Structure with a compelling headline, intro, body sections, and conclusion
- Aim for 800-1200 words


Do NOT ask for feedback, offer revisions, or include any commentary after the article.
Just deliver the finished article in markdown format.
"""
writer_agent=Agent(
    name="Writer Agent",
    instructions=WRITER_AGENT_PROMPT,
    model=MODEL
)

In [7]:
from agents import handoff

ORCHESTRATOR_AGENT_PROMPT = """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article.
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents.
Your tools and agents are specialists and should be doing the work, you are the manager.


Your process:
1. Use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
2. Pick the best research brief out of the two. Do not combine them, just pick the best one.
3. Hand off to the Writer Agent with the best research brief so it can write the article.


Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.
Do not write the article yourself, you MUST hand off to the Writer Agent.

"""
orchestrator_agent=Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model=MODEL,
    tools=[research_agent.as_tool(max_turns=30,tool_name="research_agent", tool_description="Research a topic and return a brief with key facts, statistics, themes and source URLs. Pass the topic as input.")],
    handoffs=[handoff(agent=writer_agent)]
)

Step 3: Input Guardrail

In [28]:
from agents import input_guardrail, GuardrailFunctionOutput

INPUT_GUARDRAIL_AGENT_PROMPT = """
Determine if the request involves BLOCKED topics:
- politics
- religion
- gambling
- weapons
Respond with ONLY "PASS" or "FAIL: <reason>". Nothing else.
"""
input_guardrail_agent = Agent(
    name="Input Guardrail Agent",
    instructions=INPUT_GUARDRAIL_AGENT_PROMPT,
    model= MODEL
)

@input_guardrail
async def topic_check(context, agent, input):
  result = await Runner.run(input_guardrail_agent, input = input)
  is_blocked = result.final_output.startswith("FAIL")
  print(f" {'\u274C' if is_blocked else '\u2705'} Topic check: {result.final_output}")
  return GuardrailFunctionOutput(output_info=result.final_output, tripwire_triggered=False) #is_blocked)

In [32]:
from agents import output_guardrail, GuardrailFunctionOutput

OUTPUT_GUARDRAIL_AGENT_PROMPT = """Determine if the output involves BLOCKED topics:
- politics
- religion
- gambling
- weapons
- psicological advice

Respond with ONLY "PASS" or "FAIL:<reason>". Nothing else.
"""

output_guardrail_agent = Agent(
    name="Output Guardrail Agent",
    instructions=OUTPUT_GUARDRAIL_AGENT_PROMPT,
    model=MODEL
)

@output_guardrail
async def article_check(context,agent,output):
  result = await Runner.run(output_guardrail_agent,input=output)
  is_blocked = result.final_output.startswith("FAIL")
  print(f" {'\u274C' if is_blocked else '\u2705'} Topic check: {result.final_output}")
  return GuardrailFunctionOutput(output_info=result.final_output,tripwire_triggered=is_blocked)

In [33]:
# Update the Orchestrator Agent to include the input guardrail and the output guardrail
orchestrator_agent.input_guardrails = [topic_check]
orchestrator_agent.output_guardrails= [article_check]

In [35]:
result = await Runner.run(
    orchestrator_agent,
    input="Topic: Who will win the next US presidential election?",
    max_turns=30
)

 ❌ Topic check: FAIL: politics
  ✅ search_web: Got results for Next US presidential election predictions major party candidates current political climate expert analysis
  ✅ search_web: Got results for Next US presidential election key candidates polling data election forecasts 2024
  ✅ search_web: Got results for Next US presidential election 2024 predictions major party candidates current political climate expert analysis
  ✅ fetch_url: Got 1175 chars from https://www.270towin.com/2024-presidential-election-polls/
  ✅ fetch_url: Got 5854 chars from https://www.bbc.com/news/articles/cj4x71znwxdo
  ✅ fetch_url: Got 8569 chars from https://abcnews.com/538/538s-final-forecasts-2024-election/s
  ✅ fetch_url: Got 1175 chars from https://www.270towin.com/2024-presidential-election-polls/
  ✅ fetch_url: Got 3137 chars from https://www.nytimes.com/interactive/2023/us/politics/preside
  ✅ fetch_url: Got 287002 chars from https://en.wikipedia.org/wiki/Second_presidency_of_Donald_Tr


In [16]:
display(Markdown(result.final_output))

```markdown
# Who Will Win the Next U.S. Presidential Election? A Deep Dive into the 2024 Race and What It Means for the Future

The 2024 U.S. presidential election was a nail-biter that kept the nation on edge until the final tally. With former President Donald Trump beating Vice President Kamala Harris in a high-stakes contest that fractured public opinion and reflected deep divisions in the country, the question on everyone’s minds is: who will win the next U.S. presidential election? In this deep dive, we’ll break down what happened in 2024, the key factors that swayed voters, and what these trends might mean for the upcoming elections.

---

## The 2024 Election Recap: Trump’s Comeback

The 2024 presidential election took place on November 5, 2024. The stakes were incredibly high with:

- **Republican Ticket:** Former President Donald Trump and Ohio Junior Senator JD Vance
- **Democratic Ticket:** Incumbent Vice President Kamala Harris and Minnesota Governor Tim Walz

President Joe Biden initially ran for a second term but withdrew around mid-2024, endorsing Harris as the Democratic nominee. Despite widespread expectations of a competitive race, Trump ultimately clinched the presidency with 312 electoral votes versus Harris’ 226, also winning the popular vote with a 49.8% plurality.

This victory marked history: Trump became the first president since Grover Cleveland to win non-consecutive terms, and the first Republican to win the popular vote since George W. Bush in 2004.

---

## What Drove Voters’ Decisions? Key Factors Unpacked

Understanding voter motivation is critical to projecting future election outcomes. The 2024 election revealed several prominent themes:

### 1. The Economy: Inflation and Cost of Living

Economic anxiety was the decisive issue for many voters, especially Trump supporters. A staggering **79%** of Trump voters cited rising costs and inflation as the most critical factor influencing their choice. Price hikes affected everyday groceries, housing, and energy bills and fueled a desire for “retribution” against perceived mismanagement.

Harris supporters, meanwhile, were concerned about economic issues but also placed significant weight on social and democratic principles.

### 2. Immigration: A Flashpoint Issue

Immigration ranked as a top concern, particularly on the Republican side. Reportedly, **82%** of Trump supporters viewed immigration as a “very important” issue—a central narrative of Trump’s campaign focusing on an “America First” vision and stricter border controls.

While immigration was less urgent for Harris voters, it remained an important conversation point, albeit overshadowed by healthcare and Supreme Court appointments.

### 3. Democracy and Governance: Trust on the Line

The state of American democracy was a pressing concern, especially for Democrats. Roughly **70%** of Harris voters feared that Trump might abuse government power to target political enemies, underscoring high levels of polarization and distrust.

Voters on both sides grappled with questions about election integrity, political violence, and foreign interference, further intensifying political tensions.

### 4. Abortion Rights Matter

Following the Supreme Court’s overturning of Roe v. Wade, abortion became a defining issue. About **67%** of Harris supporters prioritized abortion rights as a key voting issue. Conversely, many Trump voters supported more conservative stances on abortion, aligning with his campaign’s social policy platform.

---

## Election Environment and Challenges

The 2024 campaign was characterized by:

- Sharp political polarization and ongoing legal battles involving Trump.
- Efforts in some Republican-led states to restrict voting access and purge voter rolls.
- An alarming rise in threats to election workers and incidents of politically motivated violence, including two assassination attempts on Trump. 

The backdrop was one of heightened stakes and anxiety, with both campaigns mobilizing their bases amid fears of election disruption or contested results.

---

## The Election Outcome: What It Tells Us

Trump’s victory was driven by several intertwined factors:

- Economic dissatisfaction, especially with inflation and daily costs.
- Strong voter emphasis on immigration control.
- Gains among minority and younger voters compared to his previous runs.
- Effective ground game in crucial swing states, flipping battlegrounds like Pennsylvania, Georgia, and Michigan.

Harris struggled to capture less politically engaged voters and faced the headwinds of anti-incumbency sentiment and economic concerns.

---

## Looking Ahead: What This Means for Future Elections

The 2024 election results and key voter concerns provide a useful lens to assess who might win the next presidential election:

- **Economic Issues Will Remain Central.** Inflation, housing, healthcare costs, and economic inequality dominate voter priorities, influencing party campaigns and platforms.
  
- **Immigration Will Continue to Be a Divisive Topic.** Both parties will refine messaging around border security, immigration reform, and national identity—issues that resonate strongly with their respective bases.
  
- **Democracy and Governance Will Shape Voter Trust.** Concerns about fairness, threats to democratic institutions, and political violence may affect voter turnout and engagement.
  
- **Social Issues Like Abortion Will Influence Key Voter Blocks.** Ongoing debates will likely sway women voters, younger demographics, and independents.

---

## Conclusion: Predicting the Next Winner? It’s Complicated

While no one can predict the next president with certainty, the 2024 election underscores some critical truths:

- Candidates addressing economic realities with clear, credible plans gain voter trust.
- Clear stances on immigration and social issues mobilize core supporters.
- Voter confidence in democratic processes remains a fundamental factor.

The Republican victory in 2024 suggests that the GOP’s messaging on economy and immigration struck a chord in a nation grappling with economic hardship and social change. However, shifting demographics and emerging political landscapes mean the next race could bring surprises.

For now, political watchers will be closely monitoring potential candidates, voter sentiment, and issue salience leading into the 2028 election. One thing is sure: the battle for America’s future will continue to be fiercely contested.

---

### Sources and Further Reading

- [2024 United States presidential election - Wikipedia](https://en.wikipedia.org/wiki/2024_United_States_presidential_election)
- [Understanding the 2024 Election — PRRI Spotlight](https://prri.org/spotlight/understanding-the-2024-election-uncovering-the-key-factors-influencing-americans-presidential-votes/)
- [Issues and the 2024 Election — Pew Research Center](https://www.pewresearch.org/politics/2024/09/09/issues-and-the-2024-election/)
- [2024 Democratic Party presidential candidates - Wikipedia](https://en.wikipedia.org/wiki/2024_Democratic_Party_presidential_candidates)

---

*Stay tuned as the political story unfolds!*
```